In [2]:
import pandas as pd
import numpy as np
import duckdb


# 1. 准备模拟数据：员工系统登录日志
np.random.seed(42)
data_size = 5

login_times = [
    "2026-06-01 08:30:15.123456+08:00",
    "2026-06-15 12:45:00.987654+08:00",
    "2026-06-20 18:15:30.000000+08:00",
    "2026-06-25 23:59:59.123000+08:00",
    "2026-06-27 06:00:00.456789+08:00"
]

df_login_log = pd.DataFrame({
    'log_id': range(101, 101 + data_size),
    'user_id': [f"user_{i}" for i in range(data_size)],
    'login_timestamp': pd.to_datetime(login_times)
})

# 显示生成的数据
print("--- 原始数据集 df_login_log ---")
print(df_login_log)
print("\n数据类型查看：")
print(df_login_log.dtypes)

--- 原始数据集 df_login_log ---
   log_id user_id                  login_timestamp
0     101  user_0 2026-06-01 08:30:15.123456+08:00
1     102  user_1 2026-06-15 12:45:00.987654+08:00
2     103  user_2        2026-06-20 18:15:30+08:00
3     104  user_3 2026-06-25 23:59:59.123000+08:00
4     105  user_4 2026-06-27 06:00:00.456789+08:00

数据类型查看：
log_id                                 int64
user_id                               object
login_timestamp    datetime64[ns, UTC+08:00]
dtype: object


## 🛠️ 双轨道实战挑战

### 🚨 核心对照规则
* **SQL 轨道**：请使用 PostgreSQL 语法编写查询。
* **Pandas 轨道**：使用 Pandas 链式调用（`.assign()`）实现完全相同的输出结果。

---

### 题目 1：获取系统当前高精度时间
* **SQL 任务**：选择 `log_id`，并新增一列显示系统当前的完整时间戳（带时区，微秒级）。
* **Pandas 任务**：在 `df_login_log` 上使用链式调用，增加 `current_time_tz` 列。







In [25]:
# 题目1
# SQL轨道

query1 = """
SELECT  log_id,
        NOW() AS current_time_tz
FROM df_login_log
"""
df_result1_sql = duckdb.execute(query1).fetchdf()
print(df_result1_sql)

   log_id                  current_time_tz
0     101 2026-06-27 16:31:02.544589+08:00
1     102 2026-06-27 16:31:02.544589+08:00
2     103 2026-06-27 16:31:02.544589+08:00
3     104 2026-06-27 16:31:02.544589+08:00
4     105 2026-06-27 16:31:02.544589+08:00


In [26]:
# 题目1
# PANDAS轨道
anchor_time = pd.Timestamp.now(tz='Asia/Shanghai')
df_result1_pd = (
    df_login_log
    .loc[:,['log_id']]
    .assign(
        current_time_tz = anchor_time
    )
)
print(df_result1_pd)

   log_id                  current_time_tz
0     101 2026-06-27 16:31:05.546754+08:00
1     102 2026-06-27 16:31:05.546754+08:00
2     103 2026-06-27 16:31:05.546754+08:00
3     104 2026-06-27 16:31:05.546754+08:00
4     105 2026-06-27 16:31:05.546754+08:00


### 题目 2：去除时区的安全转换（Casting）
* **SQL 任务**：使用两种不同的转换语法（`::` 和 `CAST`），将系统当前时间转换为**不带时区**的时间戳。
* **Pandas 任务**：在 Pandas 中获取不带时区的当前时间戳，并观察两轨道的类型对齐。

In [28]:
# 题目2
# SQL轨道
# 方法1
# query2 = """
# SELECT  log_id,
#         current_time_tz :: TIMESTAMP AS current_time
# FROM df_result1_sql
# """
# df_result2_sql = duckdb.execute(query2).fetchdf()

# 方法2
query2 = """
SELECT  log_id,
        CAST(current_time_tz AS TIMESTAMP) AS current_time
FROM df_result1_sql
"""
df_result2_sql = duckdb.execute(query2).fetchdf()
print(df_result2_sql)

   log_id               current_time
0     101 2026-06-27 16:31:02.544589
1     102 2026-06-27 16:31:02.544589
2     103 2026-06-27 16:31:02.544589
3     104 2026-06-27 16:31:02.544589
4     105 2026-06-27 16:31:02.544589


In [ ]:
# 题目2
# PANDAS轨道
df_result2_pd = (
    df_result1_pd
    .assign(
        current_time_tz = lambda x:x['current_time_tz'].dt.tz_localize(None) # 去掉原有时间戳的时区.dt.tz_localize(None)
    )
)
print(df_result2_pd)

   log_id            current_time_tz
0     101 2026-06-27 16:31:05.546754
1     102 2026-06-27 16:31:05.546754
2     103 2026-06-27 16:31:05.546754
3     104 2026-06-27 16:31:05.546754
4     105 2026-06-27 16:31:05.546754


### 题目 3：限制时间戳精度
* **SQL 任务**：使用 `CURRENT_TIMESTAMP` 函数，要求返回的当前时间**秒后面不保留小数（精度为 0）**。
* **Pandas 任务**：利用 Pandas 的时间戳格式化或四舍五入方法（提示：`.round('s')`），使输出与 SQL 精度一致。

In [35]:
# 题目3
# SQL轨道
query3 = """
SELECT date_trunc('second',CURRENT_TIMESTAMP)::TIMESTAMP AS current_time_sec
"""
df_result3_sql = duckdb.execute(query3).fetchdf()
print(df_result3_sql)

     current_time_sec
0 2026-06-27 16:58:01


In [34]:
# 题目3
# PANDAS轨道
df_result3_pd = pd.Timestamp.now().round('s')
print(df_result3_pd)

2026-06-27 16:57:10


### 题目 4：计算至今生存时间（综合复习）
* **SQL 任务**：计算登录时间（`login_timestamp`）距离当前系统日期（不含时分秒）之间相差的**绝对天数**。
* **Pandas 任务**：使用 `.dt.normalize()` 或 `.dt.days` 准确计算出绝对天数，注意时区对齐问题。

In [40]:
# 题目4
# SQL轨道
query4 = """
WITH current_date_table AS (
SELECT  log_id,
        CAST(NOW()::TIMESTAMP AS DATE) AS current_date,
        CAST(login_timestamp::TIMESTAMP AS DATE) AS login_date
FROM df_login_log
)
SELECT  log_id,
        current_date - login_date AS apart_days
FROM current_date_table
"""
df_result4_sql = duckdb.execute(query4).fetchdf()
print(df_result4_sql)

   log_id  apart_days
0     101          26
1     102          12
2     103           7
3     104           2
4     105           0


In [46]:
# 题目4
# PANDAS轨道
today_achor = pd.Timestamp.now(tz='Asia/Shanghai').normalize()
df_result4_pd = (
    df_login_log
    .assign(
        apart_days = lambda x:(today_achor-x['login_timestamp'].dt.normalize() ).dt.days
    )
    [['log_id','apart_days']]
)
print(df_result4_pd)

   log_id  apart_days
0     101          26
1     102          12
2     103           7
3     104           2
4     105           0
